# 02 · Preprocessing & Feature Engineering

Pivot the long-format EIA data into a wide time-series, clean it, and add features used by the forecasting models.

In [43]:
import pandas as pd
import numpy as np

## 2.1 Load raw data

In [44]:
df = pd.read_csv(
    "../data/raw/eia_hourly_load.csv",
    parse_dates=["timestamp"]
)

## 2.2 Pivot from long to wide format

Each `type` code (D, DF, NG, TI) becomes its own column, indexed by `timestamp`.  
This is the standard reshape for multi-series EIA data.

In [45]:
df_pivot = df.pivot_table(
    index="timestamp",
    columns="type",
    values="load_mw"
).reset_index()
print("Columns after pivot:", df_pivot.columns.tolist())
df_pivot.head()

Columns after pivot: ['timestamp', 'D', 'DF', 'NG', 'TI']


type,timestamp,D,DF,NG,TI
0,2023-01-01 00:00:00,88145.0,90038.0,92327.0,4181.0
1,2023-01-01 01:00:00,85805.0,88353.0,89575.0,3770.0
2,2023-01-01 02:00:00,83630.0,85350.0,87405.0,3776.0
3,2023-01-01 03:00:00,81173.0,81746.0,84934.0,3762.0
4,2023-01-01 04:00:00,79095.0,78010.0,83080.0,3984.0


## 2.3 Sort, de-duplicate, and forward-fill

- **Sort** by timestamp to ensure chronological order before any rolling operations  
- **Drop duplicates** on the timestamp index  
- **Forward-fill** any isolated missing hours (gap-fill, not imputation of large stretches)

In [46]:
df_pivot = df_pivot.sort_values("timestamp").reset_index(drop=True)
dupes = df_pivot.duplicated().sum()
print(f"Duplicate rows removed: {dupes}")
df_pivot = df_pivot.drop_duplicates()
df_pivot = df_pivot.ffill()
print(f"Remaining nulls after ffill:\n{df_pivot.isnull().sum()}")

Duplicate rows removed: 0
Remaining nulls after ffill:
type
timestamp    0
D            0
DF           0
NG           0
TI           0
dtype: int64


## 2.4 Calendar features

Extract granular time components for use as model regressors.

In [47]:
df_pivot["hour"]      = df_pivot["timestamp"].dt.hour
df_pivot["dayofweek"] = df_pivot["timestamp"].dt.dayofweek
df_pivot["month"]     = df_pivot["timestamp"].dt.month
df_pivot["year"]      = df_pivot["timestamp"].dt.year

## 2.5 Weekend flag

Binary indicator: 1 = Saturday/Sunday, 0 = weekday. Weekend demand profiles differ significantly from weekdays.

In [48]:
df_pivot["is_weekend"] = df_pivot["dayofweek"].isin([5, 6]).astype(int)
print(df_pivot["is_weekend"].value_counts())

is_weekend
0    890
1    360
Name: count, dtype: int64


## 2.6 Season encoding

Map months to meteorological seasons (0=Winter, 1=Spring, 2=Summer, 3=Autumn).

In [49]:
season_map = {12: 0, 1: 0, 2: 0,   # Winter
              3: 1, 4: 1, 5: 1,   # Spring
              6: 2, 7: 2, 8: 2,   # Summer
              9: 3, 10: 3, 11: 3} # Autumn
df_pivot["season"] = df_pivot["month"].map(season_map)
print(df_pivot["season"].value_counts().sort_index())

season
0    1250
Name: count, dtype: int64


## 2.7 Cyclical hour encoding

Sine/cosine transform on the hour of day so models see hour 23 and hour 0 as adjacent rather than far apart.

In [50]:
df_pivot["hour_sin"] = np.sin(2 * np.pi * df_pivot["hour"] / 24)
df_pivot["hour_cos"] = np.cos(2 * np.pi * df_pivot["hour"] / 24)

## 2.8 Rolling 7-day average

168-hour rolling mean on actual demand (D) — captures weekly trend and smooths noise.  
Rows without a full 7-day lookback are dropped.

In [51]:
df_pivot["rolling_7d_avg"] = df_pivot["D"].rolling(window=24*7).mean()
before = len(df_pivot)
df_pivot = df_pivot.dropna()
print(f"Rows dropped (rolling window warm-up): {before - len(df_pivot)}")
print(f"Final dataset: {len(df_pivot):,} rows")

Rows dropped (rolling window warm-up): 167
Final dataset: 1,083 rows


## 2.9 Export to Parquet

Parquet is faster to read and ~4× smaller than CSV for this kind of wide numeric table.

In [52]:
df_pivot.to_parquet("../data/processed/pjm_clean.parquet", index=False)
print("Saved → data/processed/pjm_clean.parquet")
df_pivot.head()

df.reset_index().to_csv("../dashboard/pjm_clean_for_pbi.csv", index=False)

Saved → data/processed/pjm_clean.parquet
